## Draw Theoretical Fatality Capture AUC Curve

In [ ]:
import numpy as np
import os
import matplotlib.pyplot as plt
from matplotlib.patches import Polygon
from scipy.interpolate import make_interp_spline

from eda_toolkit import ensure_directory

## Set Up and Ensure Directories

In [ ]:
base_path = os.path.join(os.pardir)
mlflow_model_path = (
    "/home/lshpaner/Python_Projects/acled_ukraine/mlruns/models/306762030449779565"
)
# create image paths
image_path_png = os.path.join(base_path, "images", "png_images")
image_path_svg = os.path.join(base_path, "images", "svg_images")
data_path = os.path.join(base_path, "data")

# Use the function to ensure'data' directory exists
ensure_directory(image_path_png)
ensure_directory(image_path_svg)
ensure_directory(data_path)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))

# Curve
x_pts = np.array([0.0, 0.167, 0.333, 0.5, 0.667, 0.833, 1.0])
y_pts = np.array([0.0, 0.53, 0.70, 0.81, 0.89, 0.96, 1.0])
spl = make_interp_spline(x_pts, y_pts, k=3)
x_fine = np.linspace(0, 1, 400)
y_fine = spl(x_fine)

## Shaded AUC region
ax.fill_between(x_fine, 0, y_fine, color="#1D9E75", alpha=0.10, zorder=1)

################################################################################
## One highlighted trapezoid (strip k)
################################################################################

k = 2
x0, x1 = x_pts[k], x_pts[k + 1]
y0, y1 = float(spl(x0)), float(spl(x1))

trap = Polygon(
    [(x0, 0), (x1, 0), (x1, y1), (x0, y0)],
    closed=True,
    facecolor="#AEC9E8",
    alpha=0.55,
    edgecolor="#2A6096",
    linewidth=1.2,
    zorder=2,
)
ax.add_patch(trap)

# Dashed reference lines
ax.plot([0, x0], [y0, y0], color="#2A6096", lw=0.9, ls="--", zorder=3)
ax.plot([0, x1], [y1, y1], color="#2A6096", lw=0.9, ls="--", zorder=3)
ax.plot([x0, x0], [0, y0], color="#2A6096", lw=0.9, ls=":", zorder=3)
ax.plot([x1, x1], [0, y1], color="#2A6096", lw=0.9, ls=":", zorder=3)

# Y-axis labels
ax.text(
    -0.03,
    y0,
    r"$F(x_{k-1})$",
    ha="right",
    va="center",
    color="#2A6096",
    fontsize=9,
)
ax.text(
    -0.03,
    y1,
    r"$F(x_k)$",
    ha="right",
    va="center",
    color="#2A6096",
    fontsize=9,
)

# X-axis labels
ax.text(
    x0,
    -0.05,
    r"$x_{k-1}$",
    ha="center",
    va="top",
    color="#2A6096",
    fontsize=9,
)
ax.text(
    x1,
    -0.05,
    r"$x_k$",
    ha="center",
    va="top",
    color="#2A6096",
    fontsize=9,
)

# Area_k label
ax.text(
    (x0 + x1) / 2,
    (y0 + y1) / 2 - 0.05,
    r"Area$_k$",
    ha="center",
    va="center",
    color="#1A4A6B",
    fontsize=9,
    style="italic",
)

################################################################################
#### Capture curve
################################################################################

ax.plot(
    x_fine,
    y_fine,
    color="#1D9E75",
    lw=2.2,
    label="Capture curve $F(x)$",
    zorder=4,
)

################################################################################
#### Random baseline
################################################################################

ax.plot(
    [0, 1],
    [0, 1],
    color="#888780",
    lw=1.3,
    ls=(0, (7, 4)),
    label="Random baseline (AUC = 0.5)",
    zorder=3,
)

################################################################################
#### AUC annotation
################################################################################

ax.text(
    0.15,
    0.35,
    "AUC",
    color="#1D9E75",
    fontsize=11,
    fontweight="bold",
    ha="center",
)
ax.text(
    0.15,
    0.28,
    r"$=\int_0^1 F(x)\,dx$",
    color="#1D9E75",
    fontsize=9,
    ha="center",
)

################################################################################
#### Axes
################################################################################

ax.set_xlim(-0.06, 1.05)
ax.set_ylim(-0.10, 1.06)
ax.set_xticks([0, 0.25, 0.50, 0.75, 1.00])
ax.set_yticks([0, 0.25, 0.50, 0.75, 1.00])
ax.set_xlabel("Fraction of events inspected, $x$", fontsize=11)
ax.set_ylabel("Fraction of fatalities captured, $F(x)$", fontsize=11)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.tick_params(labelsize=9)
ax.grid(True, alpha=0.2, linestyle="--", linewidth=0.5)
ax.legend(loc="lower right", fontsize=9, framealpha=0.9, edgecolor="#cccccc")

plt.tight_layout()
plt.savefig(
    os.path.join(image_path_png, "capture_curve_trapezoidal.png"),
    dpi=300,
    bbox_inches="tight",
    facecolor="white",
)
plt.savefig(
    os.path.join(image_path_svg, "capture_curve_trapezoidal.svg"),
    dpi=300,
    bbox_inches="tight",
    facecolor="white",
)

plt.show()